# Voice2Reel — بيانات رسم الـ Stickman وحلقة قياس on CPU

هذا الدفتر **لا يحتاج GPU**؛ النماذج هنا صغيرة وتعمل على CPU. الهدف منه إعادة توليد وقياس corpus رسم الـ Stickman المؤلف من **60,000 خطة متجهة** (50,000 تدريب / 5,000 تحقق / 5,000 اختبار)، ثم عرض عيّنات مرسومة بالـ renderer المتجهي الحالي.

**حدود مهمة:** البيانات اصطناعية مؤلفة داخل المستودع (CC0-1.0)، بدون مراجعة خبراء، وليست صوراً ولا مجموعة تدريب diffusion. التقسيمات تُعلّم 18 نقطة وصل، 14 ميزة، و13 خطأً مرئياً مُسمّى. `0.9299` هو قياس CPU على هذه التقسيمات الاصطناعية فقط، وليس ادّعاءً بجودة الفيديو النهائي، ولا بجمال الرسم لدى مشاهدين حقيقيين، ولا بتحسّن كاتب الإنتاج `Qwen3-4B`. تدريب الكاتب على GPU يبقى في `05_quality_loop.ipynb`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/content/Voice2Reel')
if not (REPO / 'pyproject.toml').is_file():
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
        REPO = REPO.parent
assert (REPO / 'pyproject.toml').is_file(), 'Run from the repository root or upload the repository ZIP'
sys.path.insert(0, str(REPO))
WORK = REPO / 'examples' / 'stickman_drawing'
WORK.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)

## تثبيت الحزم

نثبّت الحزمة نفسها، `numpy` (تابع أساسي)، و`Pillow`/`arabic-reshaper` لرسم العيّنات العربية. لا نغيّر أوزاناً ولا ننزّل نموذج لغوي.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
     'Pillow==13.0.0', 'arabic-reshaper==3.1.0'],
    cwd=REPO, check=True,
)
print('Dependencies ready')

## إعادة التوليد بايت-ببايت وفحص كل البيانات

`--check` يعيد بناء corpus ويقارن البايتات بالملفات المحفوظة. `check_data.py` يفحص تقسيمات v2 وstickman، عدم التسريب، تطابق الـ manifest والمرآة، ويعيد حساب تقرير القياس ويطابقه مع الملف المحفوظ.

In [ ]:
build = subprocess.run(
    [sys.executable, 'scripts/build_stickman_drawing_data.py', '--check'],
    cwd=REPO, check=True, capture_output=True, text=True,
)
print(build.stdout)
subprocess.run([sys.executable, 'scripts/check_data.py'], cwd=REPO, check=True)

## ثلاث دورات موثّقة: بيانات/ميزات أكثر ← قياس أعلى

كل دورة تُ fitted نموذج ridge للتنبؤ بنقاط الوصل، ونموذج حدودي لاكتشاف الأخطاء، ثم تُقاس على `validation`. بعد اختيار أفضل دورة يُقاس `test` مرة واحدة فقط.

In [ ]:
report_path = WORK / 'artist_rounds.json'
subprocess.run(
    [sys.executable, '-m', 'voice2reel.stickman_artist', '--output', report_path],
    cwd=REPO, check=True,
)
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Validation rounds:', [item['score'] for item in report['rounds']])
print('Best round:', report['best_round'])
print('Held-out test:', report['final_test_on_best_round'])
print('Integrity checks:', len(report['integrity']), 'all true:', all(report['integrity'].values()))

## عيّنات متجهية: سليمة وأخطاء مُسمّاة

الصور أدناه مرسومة بـ `voice2reel/stickman_artist.py` من خطط المتجهات، وليست صوراً مولّدة بنموذج diffusion. العيّنات الثمانية تغطي ثلاث إيماءات سليمة وخمسة أخطاء: وصل مقطوع، نسب خاطئة، خروج من الإطار، رأس مقطوع، وتداخل مع اللوحة.

In [ ]:
from IPython.display import Image, display

samples_dir = WORK / 'samples'
subprocess.run(
    [sys.executable, '-c',
     'from voice2reel.stickman_artist import render_samples; '
     f'render_samples({str(samples_dir)!r})'],
    cwd=REPO, check=True,
)
samples = json.loads((samples_dir / 'samples.json').read_text(encoding='utf-8'))
for name in ('sample_00_present.png', 'sample_03_disconnected_joint.png',
             'sample_04_bad_proportions.png', 'sample_07_overlap.png'):
    print(name)
    display(Image(filename=str(samples_dir / name)))

## ملخص صادق وما بعده

- تم توليد corpus كبير قابل لإعادة البناء، وتم **تدريب نموذجين صغيرين على CPU** وقياسهما فعلياً: تحسّن `validation` من `0.7615` إلى `0.9304`، و`test` النهائي `0.9299`.
- هذا **يُحسّن بوابة اكتشاف أخطاء الرسم والتنبؤ بنقاط الوصل** داخل هذه البيانات، ولا يغيّر أوزان `Qwen` ولا يضيف مراجعة بشرية ولا يُثبت الذوق المهني.
- `notebooks/05_quality_loop.ipynb` ما زال مطلوباً على T4 GPU لكاتب السكربت و`final_report.json`، مع مراجعة بشرية للنصوص والصوت والفيديو.
- لو أردنا مولّد صور عصبياً لاحقاً، نحتاج مجموعة **صور raster** بتراخيص موثّقة وتدريب GPU منفصل؛ corpus المتجهات الحالي يعلّم الراسم الحتمي فقط.

In [ ]:
summary = {
    'total_rows': 60000,
    'splits': {'train': 50000, 'validation': 5000, 'test': 5000},
    'keypoints': 18,
    'features': 14,
    'defects': 13,
    'validation_scores': [item['score'] for item in report['rounds']],
    'final_test_score': report['final_test_on_best_round']['score'],
    'measured_on': 'authored synthetic CPU vector plans',
    'production_video_quality_claimed': False,
}
(WORK / 'colab_stickman_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(summary, ensure_ascii=False, indent=2))